# Structural Accessibility Boundaries and Recoverability in a Minimal H–O Redox Network Model

This notebook reproduces the computational results presented in the companion paper.

## Scope
This notebook includes:
1. Model definition
2. Baseline accessibility and threshold interpretation
3. Robustness to topology (HO₂ removal)
4. Robustness to local weight perturbations
5. Failure of global invariance under random base-weight models
6. Edge-level analysis
7. Controlled perturbation of the O₃ → O₂ return pathway
8. Emergence of a critical accessibility boundary

All edge weights represent **abstract transition cost**, not calibrated physical energies.

## 1. Imports

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import random

random.seed(0)
np.random.seed(0)

## 2. Model definition

We define a minimal directed weighted network of H–O species.  
Nodes represent oxygen-containing species. Directed edges represent allowed transitions.  
The core set is defined as: H₂O, O₂, H₃O⁺, and OH⁻.

In [ ]:
G = nx.DiGraph()

edges = [
    ("H3O+", "H2O", 1), ("H2O", "H3O+", 1),
    ("H2O", "OH-", 1), ("OH-", "H2O", 1),
    ("O2", "O2-", 2), ("O2-", "O2", 2),
    ("O2-", "HO2", 2), ("HO2", "O2-", 2),
    ("HO2", "H2O2", 2), ("H2O2", "HO2", 2),
    ("HO2", "H2O", 2), ("H2O2", "H2O", 2),
    ("OH", "H2O", 2),
    ("OH", "HO2", 3), ("HO2", "OH", 3),
    ("O2", "HO2", 2),
    ("H2O2", "O2", 2),
    ("O3", "O2", 3), ("O2", "O3", 3),
]

for u, v, w in edges:
    G.add_edge(u, v, weight=w)

core = ["H2O", "O2", "H3O+", "OH-"]

print("Nodes:", list(G.nodes()))
print("Edges:", G.number_of_edges())

## 3. Accessibility definition

Weighted shortest-path distance is interpreted as the **minimum transition cost required to reach the core**.

This provides the operational definition of recoverability used in the paper:
- a node is **recoverable** if distance ≤ threshold
- otherwise it is **non-recoverable**

In [ ]:
def weighted_distance_to_core(graph, core_nodes):
    result = {}
    for node in graph.nodes():
        costs = []
        for c in core_nodes:
            if nx.has_path(graph, node, c):
                cost = nx.shortest_path_length(graph, source=node, target=c, weight="weight")
                costs.append(cost)
        result[node] = min(costs) if costs else None
    return result

def classify_recoverability(cost, threshold=2):
    if cost is None:
        return "non-recoverable"
    return "recoverable" if cost <= threshold else "non-recoverable"

## 4. Baseline accessibility

In [ ]:
wd = weighted_distance_to_core(G, core)
wd

**Interpretation.**  
This baseline computation defines the accessibility structure of the network.  
Distances can be interpreted directly as the threshold value required for each species to remain recoverable.

## 5. Figure 1 — Accessibility structure under threshold variation

In [ ]:
avg_costs = wd.copy()
threshold_values = np.linspace(0.5, 4.0, 100)

recoverable_counts = []
for t in threshold_values:
    count = 0
    for node, cost in avg_costs.items():
        if cost is not None and cost <= t:
            count += 1
    recoverable_counts.append(count)

plt.figure(figsize=(8,5))
plt.plot(threshold_values, recoverable_counts)
plt.xlabel("Threshold")
plt.ylabel("Number of recoverable species")
plt.title("Accessibility structure under threshold variation")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figure_1_threshold_structure.png", dpi=300, bbox_inches="tight")
plt.show()

**Interpretation.**  
The recoverable set expands in discrete steps rather than continuously.  
This is consistent with a threshold-dependent accessibility structure, in which new states enter the recoverable regime only when the allowable cost exceeds specific path values.

## 6. Robustness to topology (HO₂ removal)

In [ ]:
G_top = G.copy()
G_top.remove_node("HO2")

wd_top = weighted_distance_to_core(G_top, core)

print("Baseline classification (threshold=2):")
for node, cost in wd.items():
    print(node, classify_recoverability(cost, 2))

print("\nAfter HO2 removal (threshold=2):")
for node, cost in wd_top.items():
    print(node, classify_recoverability(cost, 2))

**Interpretation.**  
Removing HO₂ does not eliminate the basic accessibility boundary at threshold = 2.  
This suggests that the observed pattern is not exclusively dependent on a single intermediate node.

## 7. Figure 2 — Robustness to local weight perturbations

In [ ]:
base_weight_dict = {(u, v): data["weight"] for u, v, data in G.edges(data=True)}

def perturb_weights(weight_dict, scale=0.2, seed=None):
    if seed is not None:
        random.seed(seed)
    new_weights = {}
    for edge, w in weight_dict.items():
        factor = random.uniform(1 - scale, 1 + scale)
        new_weights[edge] = w * factor
    return new_weights

def build_graph_from_weight_dict(base_graph, weight_dict):
    Gnew = nx.DiGraph()
    Gnew.add_nodes_from(base_graph.nodes())
    for (u, v), w in weight_dict.items():
        Gnew.add_edge(u, v, weight=w)
    return Gnew

n_runs = 200
local_results = []

for i in range(n_runs):
    wdict = perturb_weights(base_weight_dict, scale=0.2, seed=i)
    Gp = build_graph_from_weight_dict(G, wdict)
    wd_p = weighted_distance_to_core(Gp, core)
    local_results.append(wd_p)

nonrec_counts = {node: 0 for node in G.nodes()}
for res in local_results:
    for node, cost in res.items():
        if classify_recoverability(cost, threshold=2) == "non-recoverable":
            nonrec_counts[node] += 1

plt.figure(figsize=(8,4))
plt.bar(list(nonrec_counts.keys()), list(nonrec_counts.values()))
plt.ylabel("Times classified as non-recoverable")
plt.title("Local weight perturbations (±20%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figure_2_local_perturbations.png", dpi=300, bbox_inches="tight")
plt.show()

print("Non-recoverable counts under local perturbations:")
for node, count in nonrec_counts.items():
    print(f"{node}: {count}/{n_runs}")

**Interpretation.**  
Core species remain stable under local perturbations, while peripheral species vary more strongly.  
O₃ remains among the least accessible species under these tested conditions, supporting local robustness of the boundary pattern.

## 8. Figure 3 — Failure of global invariance under random base-weight models

In [ ]:
def random_base_weight_dict(base_graph, low=0.5, high=2.0, seed=None):
    if seed is not None:
        np.random.seed(seed)
    weight_dict = {}
    for u, v in base_graph.edges():
        weight_dict[(u, v)] = np.random.uniform(low, high)
    return weight_dict

n_models = 200
entry_random_models = {sp: [] for sp in G.nodes()}

for i in range(n_models):
    wdict = random_base_weight_dict(G, low=0.5, high=2.0, seed=i)
    Gnew = build_graph_from_weight_dict(G, wdict)
    wd_r = weighted_distance_to_core(Gnew, core)
    for sp in G.nodes():
        entry_random_models[sp].append(wd_r[sp])

species_to_plot = ["HO2", "H2O2", "O2-", "OH", "O3"]

plt.figure(figsize=(9,5))
for sp in species_to_plot:
    plt.hist(entry_random_models[sp], bins=20, alpha=0.4, label=sp)
plt.xlabel("Entry threshold")
plt.ylabel("Frequency")
plt.title("Entry thresholds under random base-weight models")
plt.legend()
plt.tight_layout()
plt.savefig("figure_3_random_base_weights.png", dpi=300, bbox_inches="tight")
plt.show()

o3_last_count = 0
for i in range(n_models):
    current = {sp: entry_random_models[sp][i] for sp in species_to_plot}
    ordered = sorted(current.items(), key=lambda x: x[1])
    if ordered[-1][0] == "O3":
        o3_last_count += 1

print(f"O3 entered last in {o3_last_count}/{n_models} random base-weight models")

**Interpretation.**  
Entry thresholds vary across random weight configurations, and O₃ is not always the last species to enter.  
This indicates that the accessibility boundary is not invariant under arbitrary reweighting and therefore is not a purely topological property of the network.

## 9. Figure 4 — Edge-level analysis

In [ ]:
random_models_data = []

for i in range(n_models):
    wdict = random_base_weight_dict(G, low=0.5, high=2.0, seed=i)
    Gnew = build_graph_from_weight_dict(G, wdict)
    wd_r = weighted_distance_to_core(Gnew, core)
    current = {sp: wd_r[sp] for sp in species_to_plot}
    ordered = sorted(current.items(), key=lambda x: x[1])
    o3_last = (ordered[-1][0] == "O3")
    random_models_data.append({
        "model_id": i,
        "weights": wdict,
        "entry_thresholds": wd_r,
        "o3_last": o3_last
    })

edges_list = list(base_weight_dict.keys())
o3_thresholds_all = np.array([m["entry_thresholds"]["O3"] for m in random_models_data])

edge_correlations = []
for edge in edges_list:
    vals = np.array([m["weights"][edge] for m in random_models_data])
    corr = np.corrcoef(vals, o3_thresholds_all)[0, 1]
    edge_correlations.append((edge, corr))

edge_correlations_sorted = sorted(edge_correlations, key=lambda x: abs(x[1]), reverse=True)

top_corr = edge_correlations_sorted[:10]
labels_corr = [f"{u}->{v}" for (u, v), _ in top_corr]
values_corr = [corr for _, corr in top_corr]

plt.figure(figsize=(10,5))
plt.bar(labels_corr, values_corr)
plt.xticks(rotation=90)
plt.ylabel("Correlation with O3 entry threshold")
plt.title("Top edges controlling O3 separation")
plt.tight_layout()
plt.savefig("figure_4_edge_control.png", dpi=300, bbox_inches="tight")
plt.show()

print("Top edges ranked by correlation with O3 threshold:")
for edge, corr in edge_correlations_sorted[:10]:
    print(f"{edge}: corr={corr:.3f}")

**Interpretation.**  
The dominance of the O₃ → O₂ edge suggests that accessibility boundaries may be controlled by specific return pathways rather than by global structure alone.  
This motivates the targeted perturbation experiment below.

## 10. Controlled perturbation of the O₃ → O₂ return pathway

In [ ]:
base_wdict_directed = {(u, v): d["weight"] for u, v, d in G.edges(data=True)}

def build_graph_with_controlled_o3_return(base_graph, weight_dict, new_o3_to_o2):
    Gc = nx.DiGraph()
    Gc.add_nodes_from(base_graph.nodes())
    for (u, v), w in weight_dict.items():
        if (u, v) == ("O3", "O2"):
            Gc.add_edge(u, v, weight=new_o3_to_o2)
        else:
            Gc.add_edge(u, v, weight=w)
    return Gc

o3_return_values = np.linspace(0.2, 4.0, 60)

o3_threshold_curve = []
oh_threshold_curve = []
ho2_threshold_curve = []
h2o2_threshold_curve = []
o2m_threshold_curve = []
last_intermediate_curve = []

for w_o3 in o3_return_values:
    Gc = build_graph_with_controlled_o3_return(G, base_wdict_directed, w_o3)
    wd_c = weighted_distance_to_core(Gc, core)

    o3 = wd_c["O3"]
    oh = wd_c["OH"]
    ho2 = wd_c["HO2"]
    h2o2 = wd_c["H2O2"]
    o2m = wd_c["O2-"]

    o3_threshold_curve.append(o3)
    oh_threshold_curve.append(oh)
    ho2_threshold_curve.append(ho2)
    h2o2_threshold_curve.append(h2o2)
    o2m_threshold_curve.append(o2m)
    last_intermediate_curve.append(max([oh, ho2, h2o2, o2m]))

## 11. Figure 5 — Accessibility threshold vs return-path cost

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(o3_return_values, o3_threshold_curve, label="O₃")
plt.plot(o3_return_values, last_intermediate_curve, label="Most distant intermediate")
plt.xlabel("O₃ → O₂ transition cost")
plt.ylabel("Minimum cost to reach core (H₂O)")
plt.title("Accessibility threshold vs return-path cost")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figure_5_accessibility_threshold_vs_return_cost.png", dpi=300, bbox_inches="tight")
plt.show()

**Interpretation.**  
The minimum cost required for O₃ to reach the core increases approximately linearly with the O₃ → O₂ return-path cost, while the most distant intermediate remains stable.  
This is consistent with a localized bottleneck mechanism.

## 12. Figure 6 — Emergence of accessibility boundary

In [ ]:
gap_curve = [o3_threshold_curve[i] - last_intermediate_curve[i] for i in range(len(o3_return_values))]

plt.figure(figsize=(7, 4.5))
plt.plot(o3_return_values, gap_curve)
plt.axhline(y=0, linestyle="--")
plt.xlabel("O₃ → O₂ transition cost")
plt.ylabel("Gap to accessibility boundary")
plt.title("Emergence of accessibility boundary")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figure_6_emergence_of_accessibility_boundary.png", dpi=300, bbox_inches="tight")
plt.show()

for i, g in enumerate(gap_curve):
    if g > 0:
        print("Critical transition ~", o3_return_values[i])
        break

**Interpretation.**  
The zero-crossing of the gap indicates a transition from an integrated to a separated regime.  
This supports a boundary-like behavior rather than a smooth degradation of accessibility.

## 13. Summary

This notebook reproduces the full computational logic of the paper:

- a baseline accessibility boundary appears in the minimal network
- it is locally robust to weight perturbations
- it is not globally invariant under arbitrary reweighting
- edge-level analysis identifies the O₃ → O₂ transition as the dominant controller of O₃ separation
- controlled variation of that edge produces a clean critical boundary

Overall, the results are consistent with a **bottleneck-controlled, regime-dependent** interpretation of accessibility separation in the model.